## **Incrementality Curve Analysis**

### The following analysis aims at modeling the relationship between all-possible levels of Marley Spoon's Media Spends across all various channels, brands, countries and lifecycle-types they're applied on and the estimated respective Number of Acquired Customers and Number of Reactivated Customers.

### This project will be crucial to inform the expected return of every possible level and data-cluster-distribution of media spent, which then enables Marley Spoon to optimize its media expenditure.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Relevant datasets' import and deduplication

import pandas as pd
import os

folder_path = '/content/drive/My Drive/Marketing Reports/10_LTV/Investment Committee Tooling/Input data'

media_spends    = pd.read_csv(os.path.join(folder_path, '(12650) Weekly media spends local'))
customer_counts = pd.read_csv(os.path.join(folder_path, '(12651) Weekly count of new and reactivated customers'))
acq_basket      = pd.read_csv(os.path.join(folder_path, '(12652) Weekly acquisition gross basket size'))
react_basket    = pd.read_csv(os.path.join(folder_path, '(12653) Weekly reactivation gross basket size'))
react_voucher   = pd.read_csv(os.path.join(folder_path, '(12654) Weekly reactivation unit voucher spend'))
acq_voucher     = pd.read_csv(os.path.join(folder_path, '(12655) Weekly acquisition unit voucher spend'))
season_file     = pd.read_csv(os.path.join(folder_path, 'Seasonality_Peaks_Lows - Main.csv'))

for name in ["media_spends","customer_counts","acq_basket","react_basket","react_voucher","acq_voucher","season_file"]:
    df = eval(name)
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    globals()[name] = df
    print(f"{name}: {before} → {after}  (removed {before - after} duplicates)")

media_spends: 13245 → 13245  (removed 0 duplicates)
customer_counts: 26070 → 26070  (removed 0 duplicates)
acq_basket: 1614 → 1614  (removed 0 duplicates)
react_basket: 1619 → 1619  (removed 0 duplicates)
react_voucher: 1620 → 1620  (removed 0 duplicates)
acq_voucher: 1614 → 1614  (removed 0 duplicates)
season_file: 52 → 52  (removed 0 duplicates)


In [3]:
# Data preprocessing for the purpose of successfully merging all datasets

def to_numeric_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[^\d\.\-]", "", regex=True),
                                  errors="coerce")
        else:
            print(f"[warn] numeric column not found: {c}")
    return df

def to_datetime_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
        else:
            print(f"[warn] datetime column not found: {c}")
    return df

plan = {
    "media_spends": {
        "datetime": ["01. Acquisitions and Media Acquisition Week"],
        "numeric":  ["01. Acquisitions and Media Media Spends"],
    },
    "customer_counts": {
        "datetime": ["Acquisition Week"],
        "numeric":  ["Count Acquisitions"],
    },
    "acq_basket": {
        "datetime": ["02. Customers Acquisition Week"],
        "numeric":  ["01. Orders Average Gross Basket Size"],
    },
    "react_basket": {
        "datetime": ["01. Orders Closest Reactivation Week"],
        "numeric":  ["01. Orders Average Gross Basket Size"],
    },
    "react_voucher": {
        "datetime": ["01. Orders Closest Reactivation Week"],
        "numeric":  ["01. Orders Sum of voucher value",
                     "01. Orders Count of distinct customers"],
    },
    "acq_voucher": {
        "datetime": ["02. Customers Acquisition Week"],
        "numeric":  ["01. Orders Sum of voucher value",
                     "01. Orders Count of distinct customers"],
    },
    "season_file": {
        "numeric":  season_file.columns.drop("Week number").tolist(),
    },
}

datasets = {
    "media_spends": media_spends,
    "customer_counts": customer_counts,
    "acq_basket": acq_basket,
    "react_basket": react_basket,
    "acq_voucher": acq_voucher,
    "react_voucher": react_voucher,
    "season_file": season_file,
}

for name, spec in plan.items():
    df = datasets[name]
    to_datetime_cols(df, spec.get("datetime", []))
    to_numeric_cols(df, spec.get("numeric", []))

In [4]:
# Creation of each and every data-cluster for further use

media_spends["cluster_id"] = (
    media_spends["01. Acquisitions and Media Channel Group New"] + "_" +
    media_spends["01. Acquisitions and Media Brand"] + "_" +
    media_spends["01. Acquisitions and Media Country"] + "_" +
    media_spends["01. Acquisitions and Media Lifecycle New"]
)

media_spends.head(10)

,01. Acquisitions and Media Brand,01. Acquisitions and Media Country,01. Acquisitions and Media Acquisition Week,01. Acquisitions and Media Channel Group New,01. Acquisitions and Media Lifecycle New,01. Acquisitions and Media Media Spends,cluster_id
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition
1,dn,us,2025-07-21,influencer,acquisition,2188.00,influencer_dn_us_acquisition
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition


In [5]:
# Merging of all datasets into one

masterfile = media_spends.merge(
    customer_counts,
    left_on=["01. Acquisitions and Media Brand",
             "01. Acquisitions and Media Country",
             "01. Acquisitions and Media Acquisition Week",
             "01. Acquisitions and Media Channel Group New",
             "01. Acquisitions and Media Lifecycle New"],
    right_on=["Brand",
              "Country",
              "Acquisition Week",
              "Channel Group New",
              "Lifecycle New"],
    how="left",
    suffixes=('_media_spends', '_customer_counts')
)

masterfile = masterfile.merge(
    acq_basket,
    left_on=["01. Acquisitions and Media Brand",
             "01. Acquisitions and Media Country",
             "01. Acquisitions and Media Acquisition Week",
             "01. Acquisitions and Media Lifecycle New"],
    right_on=["01. Orders Brand",
              "01. Orders Country",
              "02. Customers Acquisition Week",
              "Lifecycle"],
    how="left",
    suffixes=('', '_acq_basket')
)

masterfile = masterfile.merge(
    react_basket,
    left_on=["01. Acquisitions and Media Brand",
             "01. Acquisitions and Media Country",
             "01. Acquisitions and Media Acquisition Week",
             "01. Acquisitions and Media Lifecycle New"],
    right_on=["01. Orders Brand",
              "01. Orders Country",
              "01. Orders Closest Reactivation Week",
              "Lifecycle"],
    how="left",
    suffixes=('', '_react_basket')
)

masterfile = masterfile.merge(
    react_voucher,
    left_on=["01. Acquisitions and Media Brand",
             "01. Acquisitions and Media Country",
             "01. Acquisitions and Media Acquisition Week",
             "01. Acquisitions and Media Lifecycle New"],
    right_on=["01. Orders Brand",
              "01. Orders Country",
              "01. Orders Closest Reactivation Week",
              "Lifecycle"],
    how="left",
    suffixes=('', '_react_voucher')
)

masterfile = masterfile.merge(
    acq_voucher,
    left_on=["01. Acquisitions and Media Brand",
             "01. Acquisitions and Media Country",
             "01. Acquisitions and Media Acquisition Week",
             "01. Acquisitions and Media Lifecycle New"],
    right_on=["01. Orders Brand",
              "01. Orders Country",
              "02. Customers Acquisition Week",
              "Lifecycle"],
    how='left',
    suffixes=('', '_acq_voucher')
)

masterfile['week_number'] = masterfile['01. Acquisitions and Media Acquisition Week'].dt.isocalendar().week

masterfile = masterfile.merge(
    season_file,
    left_on=['week_number'],
    right_on=['Week number'],
    how='left',
    suffixes=('', '_season_file')
)

pd.set_option('display.max_columns', None)

masterfile.head(10)

,01. Acquisitions and Media Brand,01. Acquisitions and Media Country,01. Acquisitions and Media Acquisition Week,01. Acquisitions and Media Channel Group New,01. Acquisitions and Media Lifecycle New,01. Acquisitions and Media Media Spends,cluster_id,Unnamed: 0,Acquisition Week,Country,Brand,Lifecycle New,Channel Group New,Count Acquisitions,01. Orders Country,01. Orders Brand,02. Customers Acquisition Week,Lifecycle,01. Orders Average Gross Basket Size,01. Orders Country_react_basket,01. Orders Brand_react_basket,01. Orders Closest Reactivation Week,Lifecycle_react_basket,01. Orders Average Gross Basket Size_react_basket,Unnamed: 0_react_voucher,01. Orders Country_react_voucher,01. Orders Brand_react_voucher,01. Orders Closest Reactivation Week_react_voucher,Lifecycle_react_voucher,01. Orders Sum of voucher value,01. Orders Count of distinct customers,Unnamed: 0_acq_voucher,01. Orders Country_acq_voucher,01. Orders Brand_acq_voucher,02. Customers Acquisition Week_acq_voucher,Lifecycle_acq_voucher,01. Orders Sum of voucher value_acq_voucher,01. Orders Count of distinct customers_acq_voucher,week_number,Week number,DE seasonality Index,NL seasonality Index,BE seasonality Index,AT seasonality Index,US seasonality index,AU seasonality index
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition,22.0,2025-07-21,us,dn,acquisition,affiliate,125.0,us,dn,2025-07-21,acquisition,90.01,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,187.0,us,dn,2025-07-21,acquisition,-41089.64,596.0,30,30,-1,-1,-1,-1,-1,0
1,dn,us,2025-07-21,influencer,acquisition,2188.00,influencer_dn_us_acquisition,24.0,2025-07-21,us,dn,acquisition,influencer,16.0,us,dn,2025-07-21,acquisition,90.01,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,187.0,us,dn,2025-07-21,acquisition,-41089.64,596.0,30,30,-1,-1,-1,-1,-1,0
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition,7.0,2025-07-21,us,ms,acquisition,native,62.0,us,ms,2025-07-21,acquisition,112.22,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,1.0,us,ms,2025-07-21,acquisition,-92017.35,1017.0,30,30,-1,-1,-1,-1,-1,0
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation,31.0,2025-07-21,nl,ms,reactivation,sms,1.0,NaN,NaN,NaT,NaN,NaN,nl,ms,2025-07-21,reactivation,58.53,373.0,nl,ms,2025-07-21,reactivation,-2179.35,111.0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,30,30,-1,-1,-1,-1,-1,0
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition,20.0,2025-07-21,us,dn,acquisition,paid search non-brand,187.0,us,dn,2025-07-21,acquisition,90.01,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,187.0,us,dn,2025-07-21,acquisition,-41089.64,596.0,30,30,-1,-1,-1,-1,-1,0
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition,55.0,2025-07-21,be,ms,acquisition,affiliate,15.0,be,ms,2025-07-21,acquisition,59.78,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,1057.0,be,ms,2025-07-21,acquisition,-620.99,18.0,30,30,-1,-1,-1,-1,-1,0
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition,51.0,2025-07-21,de,ms,acquisition,affiliate,57.0,de,ms,2025-07-21,acquisition,62.56,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,715.0,de,ms,2025-07-21,acquisition,-6193.04,157.0,30,30,-1,-1,-1,-1,-1,0
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition,NaN,NaT,NaN,NaN,NaN,NaN,NaN,us,dn,2025-07-21,acquisition,90.01,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,187.0,us,dn,2025-07-21,acquisition,-41089.64,596.0,30,30,-1,-1,-1,-1,-1,0
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition,71.0,2025-07-21,au,ms,acquisition,paid search non-brand,294.0,au,ms,2025-07-21,acquisition,139.90,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,1243.0,au,ms,2025-07-21,acquisition,-189999.99,1531.0,30,30,-1,-1,-1,-1,-1,0
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition,27.0,2025-07-21,us,dn,acquisition,paid search brand,74.0,us,dn,2025-07-21,acqui

In [6]:
# Simplification of the masterfile

# Elimination of redundant columns

columns_to_drop = [
    'Unnamed: 0',
    'Acquisition Week',
    'Country',
    'Brand',
    'Lifecycle New',
    'Channel Group New',
    '01. Orders Country',
    '01. Orders Brand',
    '02. Customers Acquisition Week',
    'Lifecycle',
    '01. Orders Country_react_basket',
    '01. Orders Brand_react_basket',
    '01. Orders Closest Reactivation Week',
    'Lifecycle_react_basket',
    'Unnamed: 0_react_voucher',
    '01. Orders Country_react_voucher',
    '01. Orders Brand_react_voucher',
    '01. Orders Closest Reactivation Week_react_voucher',
    'Lifecycle_react_voucher',
    'Unnamed: 0_acq_voucher',
    '01. Orders Country_acq_voucher',
    '01. Orders Brand_acq_voucher',
    '02. Customers Acquisition Week_acq_voucher',
    'Lifecycle_acq_voucher',
    'week_number',
    'Week number'
]

masterfile = masterfile.drop(columns=columns_to_drop, errors='ignore')

# Renaming of the remaining columns

rename_dict = {
    '01. Acquisitions and Media Brand': 'Brand',
    '01. Acquisitions and Media Country': 'Country',
    '01. Acquisitions and Media Acquisition Week': 'Week',
    '01. Acquisitions and Media Channel Group New': 'Channel',
    '01. Acquisitions and Media Lifecycle New': 'Lifecycle',
    '01. Acquisitions and Media Media Spends': 'Media Spends',
    '01. Orders Average Gross Basket Size': 'Average Gross Basket Acquired Customers',
    '01. Orders Average Gross Basket Size_react_basket' : 'Average Gross Basket Reactivated Customers',
    '01. Orders Sum of voucher value': 'Sum Voucher Value Reactivated Customers',
    '01. Orders Count of distinct customers': 'Count Voucher Distinct Customers Reactivated Customers',
    '01. Orders Sum of voucher value_acq_voucher': 'Sum Voucher Value Acquired Customers',
    '01. Orders Count of distinct customers_acq_voucher': 'Count Voucher Distinct Customers Acquired Customers'
}

masterfile = masterfile.rename(columns=rename_dict)

# Unification of all seasonality columns into one general all-rows-applicable seasonality column

seasonality_map = {
    "DE": "DE seasonality Index",
    "NL": "NL seasonality Index",
    "BE": "BE seasonality Index",
    "AT": "AT seasonality Index",
    "US": "US seasonality index",
    "AU": "AU seasonality index",
}

def pick_seasonality(row):
    cc = str(row['Country']).upper()
    if cc in seasonality_map:
        return row[seasonality_map[cc]]
    return None

masterfile["seasonality"] = masterfile.apply(pick_seasonality, axis=1)

original_seasonality_columns = list(seasonality_map.values())
masterfile = masterfile.drop(columns=original_seasonality_columns, errors='ignore')

masterfile.head(10)

,Brand,Country,Week,Channel,Lifecycle,Media Spends,cluster_id,Count Acquisitions,Average Gross Basket Acquired Customers,Average Gross Basket Reactivated Customers,Sum Voucher Value Reactivated Customers,Count Voucher Distinct Customers Reactivated Customers,Sum Voucher Value Acquired Customers,Count Voucher Distinct Customers Acquired Customers,seasonality
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition,125.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1
1,dn,us,2025-07-21,influencer,acquisition,2188.00,influencer_dn_us_acquisition,16.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition,62.0,112.22,NaN,NaN,NaN,-92017.35,1017.0,-1
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation,1.0,NaN,58.53,-2179.35,111.0,NaN,NaN,-1
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition,187.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition,15.0,59.78,NaN,NaN,NaN,-620.99,18.0,-1
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition,57.0,62.56,NaN,NaN,NaN,-6193.04,157.0,-1
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition,NaN,90.01,NaN,NaN,NaN,-41089.64,596.0,-1
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition,294.0,139.90,NaN,NaN,NaN,-189999.99,1531.0,0
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition,74.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1


In [7]:
# Feature Engineering to create columns "Unit Voucher Spend Acquired Customers" and "Unit Voucher Spend Reactivated Customers"

masterfile["Unit Voucher Spend Reactivated Customers"] = (
    masterfile["Sum Voucher Value Reactivated Customers"] /
    masterfile["Count Voucher Distinct Customers Reactivated Customers"]
)

masterfile["Unit Voucher Spend Acquired Customers"] = (
    masterfile["Sum Voucher Value Acquired Customers"] /
    masterfile["Count Voucher Distinct Customers Acquired Customers"]
)

masterfile.head(10)

,Brand,Country,Week,Channel,Lifecycle,Media Spends,cluster_id,Count Acquisitions,Average Gross Basket Acquired Customers,Average Gross Basket Reactivated Customers,Sum Voucher Value Reactivated Customers,Count Voucher Distinct Customers Reactivated Customers,Sum Voucher Value Acquired Customers,Count Voucher Distinct Customers Acquired Customers,seasonality,Unit Voucher Spend Reactivated Customers,Unit Voucher Spend Acquired Customers
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition,125.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349
1,dn,us,2025-07-21,influencer,acquisition,2188.00,influencer_dn_us_acquisition,16.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition,62.0,112.22,NaN,NaN,NaN,-92017.35,1017.0,-1,NaN,-90.479204
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation,1.0,NaN,58.53,-2179.35,111.0,NaN,NaN,-1,-19.633784,NaN
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition,187.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition,15.0,59.78,NaN,NaN,NaN,-620.99,18.0,-1,NaN,-34.499444
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition,57.0,62.56,NaN,NaN,NaN,-6193.04,157.0,-1,NaN,-39.446115
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition,NaN,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition,294.0,139.90,NaN,NaN,NaN,-189999.99,1531.0,0,NaN,-124.101888
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition,74.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349


In [8]:
# Sufficiency test

# Count rows per cluster
counts = (
    media_spends.groupby("cluster_id")
    .size()
    .reset_index(name="n_obs")
)

# Apply sufficiency test (threshold = 113 rows, which is the minimum number of observations we need to diligently draw the most complex incrementality curve we'll draw per cluster)
# This would be according to the widely-applicable rule of thumb: number of observations = (10 x number of parameters estimated by the incrementality curve that estimates the most parameters per cluster) / 0.8
counts["sufficient"] = counts["n_obs"] >= 113

# Merge results back into masterfile if you want every row labeled
masterfile = masterfile.merge(counts, on="cluster_id", how="left")

# Preview results
masterfile.head(20)

,Brand,Country,Week,Channel,Lifecycle,Media Spends,cluster_id,Count Acquisitions,Average Gross Basket Acquired Customers,Average Gross Basket Reactivated Customers,Sum Voucher Value Reactivated Customers,Count Voucher Distinct Customers Reactivated Customers,Sum Voucher Value Acquired Customers,Count Voucher Distinct Customers Acquired Customers,seasonality,Unit Voucher Spend Reactivated Customers,Unit Voucher Spend Acquired Customers,n_obs,sufficient
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition,125.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,186,True
1,dn,us,2025-07-21,influencer,acquisition,2188.00,influencer_dn_us_acquisition,16.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,109,False
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition,62.0,112.22,NaN,NaN,NaN,-92017.35,1017.0,-1,NaN,-90.479204,177,True
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation,1.0,NaN,58.53,-2179.35,111.0,NaN,NaN,-1,-19.633784,NaN,148,True
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition,187.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,183,True
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition,15.0,59.78,NaN,NaN,NaN,-620.99,18.0,-1,NaN,-34.499444,185,True
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition,57.0,62.56,NaN,NaN,NaN,-6193.04,157.0,-1,NaN,-39.446115,186,True
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition,NaN,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,183,True
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition,294.0,139.90,NaN,NaN,NaN,-189999.99,1531.0,0,NaN,-124.101888,182,True
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition,74.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,186,True


In [9]:
# Build of our variables' aliases and targets to help further modeling

import numpy as np

# --------- COLUMN ALIASES -----------------------------------------------------
X_COL         = "Media Spends"
CLUSTER_COL   = "cluster_id"
LIFECYCLE_COL = "Lifecycle"
COUNT_COL     = "Count Acquisitions"

CONTROL_COLS = [
    "seasonality",
    "Unit Voucher Spend Reactivated Customers",
    "Unit Voucher Spend Acquired Customers",
    "Average Gross Basket Acquired Customers",
    "Average Gross Basket Reactivated Customers",
]

EPS = 1e-9

# --------- BUILD TARGETS (row level) -----------------------------------------
_lc = masterfile[LIFECYCLE_COL].astype(str).str.strip().str.lower()
masterfile["_Y_acq"] = np.where(_lc.eq("acquisition"), masterfile[COUNT_COL].fillna(0), 0.0)
masterfile["_Y_rea"] = np.where(_lc.eq("reactivation"), masterfile[COUNT_COL].fillna(0), 0.0)


In [10]:
# Splitting of the masterfile into 2 datasets: one designed for training our models, the other designed to test those very models

# 0) keep only clusters that passed the sufficiency check
mf = masterfile.loc[masterfile["sufficient"] == True].copy()

# 1) Per-cluster 80/20 split on Week (time-ordered)
cut_per_cluster = mf.groupby(CLUSTER_COL)["Week"].transform(lambda s: s.quantile(0.80))
mf["set"] = np.where(mf["Week"] <= cut_per_cluster, "TRAIN", "TEST")

# 2) Aggregation rules (sum targets, mean controls)
agg_dict = {"_Y_acq": "sum", "_Y_rea": "sum"}
for c in CONTROL_COLS:
    if c in mf.columns:
        agg_dict[c] = "mean"

def make_panel(df):
    return (
        df.groupby([CLUSTER_COL, X_COL], dropna=False, as_index=False)
          .agg(agg_dict)
          .rename(columns={X_COL: "X"})
    )

panel      = make_panel(mf[mf["set"] == "TRAIN"])
panel_test = make_panel(mf[mf["set"] == "TEST"])

# keep masterfile in-sync for later cells if they expect it
masterfile = mf

# Sanity checks
print({
    "clusters_total": masterfile[CLUSTER_COL].nunique(),
    "train_rows": len(panel),
    "test_rows": len(panel_test)
})

{'clusters_total': 66, 'train_rows': 7991, 'test_rows': 1985}


In [11]:
# Representativeness test (seasonality) -- run on TRAIN slice only
# Rule: representative if (p > 0.05) OR (p <= 0.05 and Cramér's V < 0.29)

from scipy.stats import chisquare

ALPHA  = 0.05
V_THR  = 0.29
cats   = [-1, 0, 1]

# === use TRAIN only ===
mf_train = masterfile.loc[masterfile["set"] == "TRAIN"].copy()

# global proportions from TRAIN
global_props = (
    mf_train["seasonality"].dropna()
    .value_counts(normalize=True)
    .reindex(cats, fill_value=0.0)
    .values
)

def rep_test(g):
    s   = g["seasonality"].dropna()
    obs = s.value_counts().reindex(cats, fill_value=0.0).astype(float).values
    n   = obs.sum()
    exp = global_props * n
    # chi-square validity guard
    if n == 0 or (exp < 5).any():
        return pd.Series({"p_value": np.nan, "chi2": np.nan, "cramers_v": np.nan})
    chi2, p = chisquare(f_obs=obs, f_exp=exp)
    k = len(cats)
    V = np.sqrt(chi2 / (n * (k - 1))) if k > 1 else np.nan
    return pd.Series({"p_value": float(p), "chi2": float(chi2), "cramers_v": float(V)})

# per-cluster test on TRAIN
rep_flags = (
    mf_train[["cluster_id","seasonality"]]
      .groupby("cluster_id", group_keys=False)
      .apply(rep_test)
      .reset_index()
)

rep_flags["rep_strict"] = rep_flags["p_value"] > ALPHA
rep_flags["rep_final"]  = np.where(
    rep_flags["p_value"].isna(), np.nan,
    (rep_flags["p_value"] > ALPHA) |
    ((rep_flags["p_value"] <= ALPHA) & (rep_flags["cramers_v"] < V_THR))
)

# attach flags back to ALL rows (train + test) by cluster_id
cols = ["p_value","chi2","cramers_v","rep_strict","rep_final"]
masterfile = masterfile.join(rep_flags.set_index("cluster_id")[cols], on="cluster_id")
masterfile["rep_final"] = masterfile["rep_final"].astype("boolean")

masterfile.head(20)

/tmp/ipython-input-581136887.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rep_test)


,Brand,Country,Week,Channel,Lifecycle,Media Spends,cluster_id,Count Acquisitions,Average Gross Basket Acquired Customers,Average Gross Basket Reactivated Customers,Sum Voucher Value Reactivated Customers,Count Voucher Distinct Customers Reactivated Customers,Sum Voucher Value Acquired Customers,Count Voucher Distinct Customers Acquired Customers,seasonality,Unit Voucher Spend Reactivated Customers,Unit Voucher Spend Acquired Customers,n_obs,sufficient,_Y_acq,_Y_rea,set,p_value,chi2,cramers_v,rep_strict,rep_final
0,dn,us,2025-07-21,affiliate,acquisition,9914.32,affiliate_dn_us_acquisition,125.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,186,True,125.0,0.0,TEST,2.060151e-02,7.764782,0.161420,False,True
2,ms,us,2025-07-21,native,acquisition,12778.84,native_ms_us_acquisition,62.0,112.22,NaN,NaN,NaN,-92017.35,1017.0,-1,NaN,-90.479204,177,True,62.0,0.0,TEST,2.745093e-02,7.190710,0.159684,False,True
3,ms,nl,2025-07-21,sms,reactivation,23.54,sms_ms_nl_reactivation,1.0,NaN,58.53,-2179.35,111.0,NaN,NaN,-1,-19.633784,NaN,148,True,0.0,1.0,TEST,6.389936e-01,0.895722,0.061607,True,True
4,dn,us,2025-07-21,paid search non-brand,acquisition,18524.47,paid search non-brand_dn_us_acquisition,187.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,183,True,187.0,0.0,TEST,1.000893e-02,9.208556,0.177584,False,True
5,ms,be,2025-07-21,affiliate,acquisition,416.70,affiliate_ms_be_acquisition,15.0,59.78,NaN,NaN,NaN,-620.99,18.0,-1,NaN,-34.499444,185,True,15.0,0.0,TEST,4.775999e-02,6.083134,0.143357,False,True
6,ms,de,2025-07-21,affiliate,acquisition,3827.68,affiliate_ms_de_acquisition,57.0,62.56,NaN,NaN,NaN,-6193.04,157.0,-1,NaN,-39.446115,186,True,57.0,0.0,TEST,5.829934e-02,5.684329,0.138112,True,True
7,dn,us,2025-07-21,other,acquisition,156.76,other_dn_us_acquisition,NaN,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,183,True,0.0,0.0,TEST,1.000893e-02,9.208556,0.177584,False,True
8,ms,au,2025-07-21,paid search non-brand,acquisition,61128.41,paid search non-brand_ms_au_acquisition,294.0,139.90,NaN,NaN,NaN,-189999.99,1531.0,0,NaN,-124.101888,182,True,294.0,0.0,TEST,3.625508e-06,25.055032,0.293933,False,False
9,dn,us,2025-07-21,paid search brand,acquisition,3946.31,paid search brand_dn_us_acquisition,74.0,90.01,NaN,NaN,NaN,-41089.64,596.0,-1,NaN,-68.942349,186,True,74.0,0.0,TEST,2.060151e-02,7.764782,0.161420,False,True
10,ms,us,2025-07-21,paid search non-brand,acquisition,33429.86,paid search non-brand_ms_us_acquisition,173.0,112.22,NaN,NaN,NaN,-92017.35,1017.0,-1,NaN,-90.479204,183,True,173.0,0.0,TEST,1.000893e-02,9.208556,0.177584,False,True


In [12]:
# Build of incrementality curves, controlling for seasonality, voucher spends and basket sizes

import json, warnings
import statsmodels.api as sm
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

warnings.filterwarnings("ignore")

# ----- small constants / guards -----
EPS = 1e-9  # to avoid log(0) / division by zero


# --------- 11 FUNCTIONAL FORMS - for curve_fit with multiple independent variables ----------
# These functions accept a tuple (X, Control1, Control2, ...) as the first argument
# and the parameters of the curve as subsequent arguments.

def poly_func(independent_vars, *params):
    p = np.asarray(independent_vars[0], dtype=float)  # Media Spends (X)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    num_controls = len(controls)

    # Parameters: intercept, b1 (for P), b2 (for P^2), ..., control coeffs...
    intercept = params[0]
    base = intercept
    param_idx = 1

    # Number of P terms (b1..bK) is whatever is left after intercept and control coeffs
    num_p_params = len(params) - 1 - num_controls
    if num_p_params > 0:
        for i in range(num_p_params):  # i=0 -> P^1, i=1 -> P^2, ...
            if param_idx < len(params):
                base += params[param_idx] * p ** (i + 1)
                param_idx += 1
            else:
                raise ValueError("Parameter count mismatch for polynomial terms")

    # Add linear terms for control variables
    for i in range(num_controls):
        if param_idx < len(params):
            base += params[param_idx] * controls[i]
            param_idx += 1
        else:
            raise ValueError("Parameter count mismatch for control terms")

    return base


def loglin_func(independent_vars, a, b, *control_params):
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    base = a + b * p
    for i in range(len(controls)):
        if i < len(control_params):
            base += control_params[i] * controls[i]
        else:
            raise ValueError("Parameter count mismatch for control terms in loglin_func")
    return np.exp(base)


def semilog_func(independent_vars, a, b, *control_params):
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    base = a + b * np.log(np.clip(p, EPS, None))
    for i in range(len(controls)):
        if i < len(control_params):
            base += control_params[i] * controls[i]
        else:
            raise ValueError("Parameter count mismatch for control terms in semilog_func")
    return base


def dlog_func(independent_vars, a, b, *control_params):
    # ln(Q) = a + b*ln(P) + Σ γ_i*Control_i   =>   Q = exp(a + Σ γ_i*Control_i) * P^b
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    base_exp_a = a
    for i in range(len(controls)):
        if i < len(control_params):
            base_exp_a += control_params[i] * controls[i]
        else:
            raise ValueError("Parameter count mismatch for control terms in dlog_func")
    return np.exp(base_exp_a) * np.clip(p, EPS, None) ** b


def expdecay_func(independent_vars, alpha, beta, k, *control_params):
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    if len(control_params) != len(controls):
        raise ValueError("Parameter count mismatch for control terms in expdecay_func")
    controls_term = sum(cp * c for cp, c in zip(control_params, controls))
    return alpha + beta * (1 - np.exp(-k * p)) + controls_term


def reciprocal_func(independent_vars, a, b, *control_params):
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    base = a + b / np.clip(p, EPS, None)
    for i in range(len(controls)):
        if i < len(control_params):
            base += control_params[i] * controls[i]
        else:
            raise ValueError("Parameter count mismatch for control terms in reciprocal_func")
    return base


def logistic_func(independent_vars, L, k, p0, *control_params):
    """
    Q = L / (1 + exp(-k * (P - (p0 + Σ γ_i·Control_i))))
    - L > 0
    - k unrestricted (sign shows direction)
    - controls shift the midpoint p0
    """
    p = np.asarray(independent_vars[0], dtype=float)
    controls = tuple(np.asarray(c, dtype=float) for c in independent_vars[1:])
    shift = 0.0
    for i in range(len(controls)):
        if i < len(control_params):
            shift += control_params[i] * controls[i]
        else:
            raise ValueError("Parameter count mismatch for control terms in logistic_func")
    eta = k * (p - (p0 + shift))
    return L / (1.0 + np.exp(-eta))


# Dictionary of forms - functions accept multiple independent variables
FORMS = {
    "linear_X_controls": (poly_func,     None, None),  # poly with degree=1 effectively
    "poly1":             (poly_func,     None, None),  # linear (P^1) + controls
    "poly2":             (poly_func,     None, None),  # up to P^2
    "poly3":             (poly_func,     None, None),
    "poly4":             (poly_func,     None, None),
    "poly5":             (poly_func,     None, None),
    "loglin":            (loglin_func,   None, None),
    "semilog":           (semilog_func,  None, None),
    "doublelog":         (dlog_func,     None, None),
    "expdecay":          (expdecay_func, None, None),
    "reciprocal":        (reciprocal_func, None, None),
    "logistic":          (logistic_func, None, None),
}


# --------- SCORING & EQUATIONS ---------------------------------------------------------
def score_fit(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    mae = mean_absolute_error(y, yhat)
    try:
        rmse = mean_squared_error(y, yhat, squared=False)
    except TypeError:
        rmse = np.sqrt(mean_squared_error(y, yhat))
    ssr = np.sum((y - yhat) ** 2)
    sst = np.sum((y - np.mean(y)) ** 2)
    r2 = 1.0 - (ssr / sst) if sst > 0 else 0.0
    return mae, rmse, r2


def human_equation(name, params=None, control_cols=None):
    """
    Render a readable equation. If params is None, return the canonical symbolic form.
    """
    # Canonical forms
    if params is None or len(params) == 0:
        canonical = {
            # Polynomials (controls enter additively)
            "poly1":  "Q = a + b·P + Σ(γ_i·Control_i)",
            "poly2":  "Q = a + b·P + c·P^2 + Σ(γ_i·Control_i)",
            "poly3":  "Q = a + b·P + c·P^2 + d·P^3 + Σ(γ_i·Control_i)",
            "poly4":  "Q = a + b·P + c·P^2 + d·P^3 + e·P^4 + Σ(γ_i·Control_i)",
            "poly5":  "Q = a + b·P + c·P^2 + d·P^3 + e·P^4 + f·P^5 + Σ(γ_i·Control_i)",

            # Linear X with controls (same as poly1)
            "linear_X_controls": "Q = a + b·P + Σ(γ_i·Control_i)",

            # Log-linear
            "loglin":    "Q = exp(a + b·P + Σ(γ_i·Control_i))",
            # Semi-log
            "semilog":   "Q = a + b·ln(P) + Σ(γ_i·Control_i)",
            # Double-log
            "doublelog": "Q = exp(a + Σ(γ_i·Control_i)) · P^b",
            # Exponential
            "expdecay":  "Q = α + β·(1 - exp(-k·P)) + Σ(γ_i·Control_i)",
            # Reciprocal
            "reciprocal":"Q = a + b/P + Σ(γ_i·Control_i)",
            # Logistic
            "logistic":  "Q = L / (1 + exp(−k·(P - (P0 + Σ(γ_i·Control_i)))))",
        }
        return canonical.get(name, name)

    # Defensive: params must be a list-like of numbers
    if not isinstance(params, (list, tuple)):
        return human_equation(name, None, control_cols)

    parts = []
    param_idx = 0
    num_controls = len(control_cols) if control_cols is not None else 0

    # Polynomial & linear_X_controls (OLS)
    if name.startswith("poly") or name == "linear_X_controls":
        # Intercept
        if len(params) <= param_idx or not isinstance(params[param_idx], (int, float, np.number)):
            return human_equation(name, None, control_cols)
        parts.append("{:.6g}".format(params[param_idx]))
        param_idx += 1

        # Number of X params before controls
        num_x_params = len(params) - 1 - num_controls
        if num_x_params > 0:
            # X^1
            if len(params) <= param_idx or not isinstance(params[param_idx], (int, float, np.number)):
                return human_equation(name, None, control_cols)
            parts.append("{:+.6g}·P".format(params[param_idx]))
            param_idx += 1
            # X^2..X^k
            for i in range(2, num_x_params + 1):
                if len(params) <= param_idx or not isinstance(params[param_idx], (int, float, np.number)):
                    return human_equation(name, None, control_cols)
                parts.append("{:+.6g}·P^{}".format(params[param_idx], i))
                param_idx += 1

        # Controls
        if num_controls > 0:
            if len(params) - param_idx != num_controls:
                return human_equation(name, None, control_cols)
            for i in range(num_controls):
                if not isinstance(params[param_idx], (int, float, np.number)):
                    return human_equation(name, None, control_cols)
                parts.append("{:+.6g}·{}".format(params[param_idx], control_cols[i]))
                param_idx += 1

        return "Q = " + " ".join(parts)

    # Non-polynomial
    if name == "loglin":
        if len(params) != 2 + num_controls:
            return human_equation(name, None, control_cols)
        a, b = params[0], params[1]
        ctrl = params[2:]
        core = ["{:.6g}".format(a), "{:+.6g}·P".format(b)]
        ctrlp = ["{:+.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]
        return "Q = exp({})".format(" ".join(core + ctrlp))

    elif name == "semilog":
        if len(params) != 2 + num_controls:
            return human_equation(name, None, control_cols)
        a, b = params[0], params[1]
        ctrl = params[2:]
        core = ["{:.6g}".format(a), "{:+.6g}·ln(P)".format(b)]
        ctrlp = ["{:+.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]
        return "Q = {}".format(" ".join(core + ctrlp))

    elif name == "doublelog":
        if len(params) != 2 + num_controls:
            return human_equation(name, None, control_cols)
        a, b = params[0], params[1]
        ctrl = params[2:]
        exp_a = ["{:.6g}".format(a)] + ["{:+.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]
        return "Q = exp({})·P^{:.6g}".format(" ".join(exp_a), b)

    elif name == "expdecay":
        if len(params) != 2 + num_controls:
            return human_equation(name, None, control_cols)
        a, b = params[0], params[1]
        ctrl = params[2:]
        core = ["{:.6g}".format(a), "{:+.6g}·P".format(b)]
        ctrlp = ["{:+.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]
        return "Q = exp({})".format(" ".join(core + ctrlp))

    elif name == "reciprocal":
        if len(params) != 2 + num_controls:
            return human_equation(name, None, control_cols)
        a, b = params[0], params[1]
        ctrl = params[2:]
        core = ["{:.6g}".format(a), "{:+.6g} / P".format(b)]
        ctrlp = ["{:+.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]
        return "Q = {}".format(" ".join(core + ctrlp))

    elif name == "logistic":
        if len(params) != 3 + num_controls:
            return human_equation(name, None, control_cols)
        L, k, p0 = params[0], params[1], params[2]
        ctrl = params[3:]
        shift = " + ".join(["{:.6g}·{}".format(ctrl[i], control_cols[i]) for i in range(num_controls)]) if num_controls else "0"
        return "Q = {:.6g} / (1 + exp({:+.6g}·(P - ({:.6g} + {}))))".format(L, -k, p0, shift)

    return name  # Fallback


# --------- FITTING (per cluster × target) ----------------------------------------------
def fit_all_forms(cluster_df, target_col, control_cols):
    """
    Fits polynomial forms (linear to poly5) using OLS with X and effective control variables.
    Attempts to fit non-polynomial forms using curve_fit with X and effective control variables.
    Dynamically excludes control columns with all NaN values for a given cluster.
    """
    results = []

    if cluster_df.empty or target_col not in cluster_df.columns or "X" not in cluster_df.columns:
        return results

    # Effective controls (have at least one non-null)
    effective_control_cols = [c for c in (control_cols or [])
                              if c in cluster_df.columns and not cluster_df[c].dropna().empty]
    num_effective_controls = len(effective_control_cols)

    # Required cols and drop rows with NaNs in them
    required_cols = ["X", target_col] + effective_control_cols
    if not all(col in cluster_df.columns for col in required_cols):
        return results

    data_for_fitting = cluster_df[required_cols].dropna().copy()
    if data_for_fitting.empty or len(data_for_fitting) < 2 + num_effective_controls:
        return results

    y = data_for_fitting[target_col].astype(float).values
    X_data = data_for_fitting["X"].astype(float).values
    Control_data = data_for_fitting[effective_control_cols].astype(float).values if num_effective_controls > 0 else np.empty((len(X_data), 0))

    # For curve_fit: tuple (X, C1, C2, ...)
    independent_vars_curve_fit = tuple([X_data] + [Control_data[:, i] for i in range(num_effective_controls)])

    # --- Fit Polynomial Forms using OLS ---
    # Need at least (#predictors + 1) observations.
    # Predictors for degree D: D (P^1..P^D) + num_controls; + intercept (handled by sm.add_constant)
    # So D <= n - num_controls - 2
    max_poly_degree_ols = int(min(5, max(1, len(data_for_fitting) - num_effective_controls - 2)))  # at least 1 (linear)

    # Always fit linear_X_controls explicitly (X + controls)
    try:
        ols_predictors = pd.DataFrame({"X": data_for_fitting["X"].values})
        for c in effective_control_cols:
            ols_predictors[c] = data_for_fitting[c].values
        X_ols = sm.add_constant(ols_predictors, has_constant="add")
        if X_ols.shape[0] >= X_ols.shape[1]:
            ols_model = sm.OLS(y, X_ols).fit()
            yhat = ols_model.fittedvalues
            mae, rmse, r2 = score_fit(y, yhat)
            lin_params = ols_model.params.tolist()
            results.append({
                "form": "linear_X_controls",
                "equation": human_equation("linear_X_controls", lin_params, effective_control_cols),
                "params_json": json.dumps([float(p) for p in lin_params]),
                "MAE": float(mae), "RMSE": float(rmse), "R2": float(r2),
                "n": int(len(y)),
                "effective_control_cols": effective_control_cols
            })
        else:
            lin_params = None
    except Exception as e:
        results.append({
            "form": "linear_X_controls", "equation": human_equation("linear_X_controls", None),
            "params_json": None, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
            "n": int(len(y)), "error": str(e),
            "effective_control_cols": effective_control_cols
        })
        lin_params = None

    # Fit poly2..poly5 (keeping names consistent: poly1 is linear, which we already fit)
    for degree in range(2, max_poly_degree_ols + 1):
        form_name = f"poly{degree}"

        try:
            ols_predictors = pd.DataFrame({"X": data_for_fitting["X"].values})
            # Add X^2..X^degree
            for i in range(2, degree + 1):  # << fixed bound so poly2 adds P^2, poly3 adds up to P^3, etc.
                ols_predictors[f"X_poly_{i}"] = data_for_fitting["X"].values ** i
            for c in effective_control_cols:
                ols_predictors[c] = data_for_fitting[c].values

            X_ols = sm.add_constant(ols_predictors, has_constant="add")
            if X_ols.shape[0] < X_ols.shape[1]:
                continue

            ols_model = sm.OLS(y, X_ols).fit()
            yhat = ols_model.fittedvalues
            mae, rmse, r2 = score_fit(y, yhat)
            params = ols_model.params.tolist()

            results.append({
                "form": form_name,
                "equation": human_equation(form_name, params, effective_control_cols),
                "params_json": json.dumps([float(p) for p in params]),
                "MAE": float(mae), "RMSE": float(rmse), "R2": float(r2),
                "n": int(len(y)),
                "effective_control_cols": effective_control_cols
            })
        except Exception as e:
            results.append({
                "form": form_name, "equation": human_equation(form_name, None),
                "params_json": None, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "n": int(len(y)), "error": str(e),
                "effective_control_cols": effective_control_cols
            })

    # --- Fit Non-Polynomial Forms using curve_fit ---
    non_poly_forms_to_fit = {
        "loglin": loglin_func,
        "semilog": semilog_func,
        "doublelog": dlog_func,
        "expdecay": expdecay_func,
        "reciprocal": reciprocal_func,
        "logistic": logistic_func,
    }

    # Use linear OLS params as a heuristic initial guess when helpful
    linear_ols_params = None
    for res in results:
        if res["form"] == "linear_X_controls" and res["params_json"] is not None:
            linear_ols_params = json.loads(res["params_json"])
            break

    expected_params_count = {
        "loglin": 2 + num_effective_controls,
        "semilog": 2 + num_effective_controls,
        "doublelog": 2 + num_effective_controls,
        "expdecay": 2 + num_effective_controls,
        "reciprocal": 2 + num_effective_controls,
        "logistic": 3 + num_effective_controls,
    }

    for form_name, func in non_poly_forms_to_fit.items():
        expected_n_params = expected_params_count[form_name]
        # default p0
        p0 = [0.0] * expected_n_params

        # heuristic from linear
        if linear_ols_params is not None and len(linear_ols_params) >= 2:
            intercept_linear = linear_ols_params[0]
            x_coeff_linear = linear_ols_params[1]
            # Take the last num_effective_controls as control coeffs
            control_coeffs_linear = linear_ols_params[-num_effective_controls:] if num_effective_controls > 0 else []

            if form_name in ["loglin", "semilog", "reciprocal", "expdecay"]:
                p0 = [intercept_linear, x_coeff_linear] + list(control_coeffs_linear)
            elif form_name == "doublelog":
                # ln(intercept) can be bad if intercept <= 0
                a0 = np.log(max(EPS, intercept_linear))
                p0 = [a0, x_coeff_linear] + list(control_coeffs_linear)
            elif form_name == "logistic":
                L0 = max(float(np.nanmax(y)), EPS)
                p00 = float(np.nanmedian(X_data)) if np.isfinite(np.nanmedian(X_data)) else float(np.mean(X_data))
                k0 = -0.01  # gentle slope by default
                p0 = [L0, k0, p00] + [0.0] * num_effective_controls

        # bounds
        if form_name == "logistic":
            lower_bounds = [EPS] + [-np.inf] * (expected_n_params - 1)
            upper_bounds = [np.inf] * expected_n_params
            bounds = (lower_bounds, upper_bounds)
        else:
            bounds = ([-np.inf] * expected_n_params, [np.inf] * expected_n_params)

        try:
            popt, pcov = curve_fit(
                func,
                independent_vars_curve_fit,
                y,
                p0=p0,
                bounds=bounds,
                maxfev=5000
            )
            yhat = func(independent_vars_curve_fit, *popt)
            mae, rmse, r2 = score_fit(y, yhat)
            params = popt.tolist()
            results.append({
                "form": form_name,
                "equation": human_equation(form_name, params, effective_control_cols),
                "params_json": json.dumps([float(p) for p in params]),
                "MAE": float(mae), "RMSE": float(rmse), "R2": float(r2),
                "n": int(len(y)),
                "effective_control_cols": effective_control_cols
            })
        except RuntimeError as e:
            results.append({
                "form": form_name, "equation": human_equation(form_name, None),
                "params_json": None, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "n": int(len(y)), "error": str(e),
                "effective_control_cols": effective_control_cols
            })
        except Exception as e:
            results.append({
                "form": form_name, "equation": human_equation(form_name, None),
                "params_json": None, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "n": int(len(y)), "error": str(e),
                "effective_control_cols": effective_control_cols
            })

    return results


# --------- RUN PER CLUSTER (apply lifecycle rule; NO selection) ------------------------
def targets_for_cluster_id(cid_str):
    s = str(cid_str).strip().lower()
    out = []
    if "acquisition" in s:
        out.append(("_Y_acq", "A_acq"))
    if "reactivation" in s:
        out.append(("_Y_rea", "B_rea"))
    # Optionally include total
    # if "acquisition" in s or "reactivation" in s:
    #     out.append(("_Y_total", "C_total"))
    return out

all_rows = []
unmatched_clusters = []

if 'panel' in locals() and isinstance(panel, pd.DataFrame) and not panel.empty:
    for cid, g in panel.groupby(CLUSTER_COL, dropna=False):
        local_targets = targets_for_cluster_id(cid)
        if not local_targets:
            unmatched_clusters.append(cid)
            continue
        for ycol, tag in local_targets:
            fits = fit_all_forms(g, ycol, CONTROL_COLS)
            for r in fits:
                r.update({CLUSTER_COL: cid, "target": tag})
            all_rows.extend(fits)

    results_df = pd.DataFrame(all_rows)
    print("All tentative fits (head):")
    display(results_df.head(50))
    if unmatched_clusters:
        print("[info] Skipped clusters with no clear lifecycle in their name:", list(unmatched_clusters))
else:
    print("[info] 'panel' DataFrame is not available or is empty. Please ensure the data loading and splitting steps were run.")


All tentative fits (head):


,form,equation,params_json,MAE,RMSE,R2,n,effective_control_cols,cluster_id,target,error
0,linear_X_controls,Q = 103.237 +0.00461661·P -1.56585·seasonality...,"[103.2373397742181, 0.004616605199437726, -1.5...",17.026101,23.262695,0.688763,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
1,poly2,Q = 102.11 +0.0054406·P -2.49976e-08·P^2 -1.27...,"[102.11031857478781, 0.005440602461672165, -2....",17.028808,23.206764,0.690257,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
2,poly3,Q = 115.138 +0.00310095·P +1.43735e-07·P^2 -3....,"[115.13780795651412, 0.003100948525312037, 1.4...",16.834269,23.134931,0.692172,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
3,poly4,Q = 6.51448e-06 +0.0204643·P -1.81023e-06·P^2 ...,"[6.51448258199669e-06, 0.020464263503491755, -...",17.395821,23.382810,0.685540,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
4,poly5,Q = 1.24365e-13 -7.12329e-08·P +3.06465e-06·P^...,"[1.2436466729364595e-13, -7.123290808238266e-0...",20.735628,26.545772,0.594713,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
5,loglin,Q = exp(4.38995 +3.72319e-05·P -0.0505899·seas...,"[4.389950356204962, 3.723193158795367e-05, -0....",18.138194,24.371224,0.658393,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
6,semilog,Q = -303.702 +52.17·ln(P) +1.68377·seasonality...,"[-303.701681918958, 52.169998831506746, 1.6837...",19.110122,25.480915,0.626576,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
7,doublelog,Q = exp(4.63703 -1.56585·seasonality -0.299261...,"[4.637030607124182, 0.004616605199437726, -1.5...",103.689189,111.759394,-6.183562,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN
8,expdecay,Q = α + β·(1 - exp(-k·P)) + Σ(γ_i·Control_i),None,NaN,NaN,NaN,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,Parameter count mismatch for control terms in ...
9,reciprocal,Q = 224.636 -276634 / P +8.20753·seasonality -...,"[224.6356892715859, -276634.224070696, 8.20753...",24.344635,31.647917,0.423948,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN


In [13]:
# results_df.to_csv("tentative_fits.csv", index=False)

# from google.colab import files
# files.download("tentative_fits.csv")

In [14]:
# === Robust Guardrails for Incrementality Curves (Spend -> Customers) ===
X_COL = "X"

# --- knobs you can tweak ---
EPS = 1e-9
X_CENTRAL_PCTS = (5, 95)           # evaluate guardrails on central band of X
GRID_MAX = 4000
DECIMAL_STEP = 0.01

DERIV_TOL = 1e-12                  # allow tiny negative dY/dX due to FP noise
MAX_NEG_DERIV_SHARE = 0.05         # ≤5% of grid may have (slightly) negative slope

ELAST_HI = 15.0                    # “reasonable” upper bound for elasticity
ELAST_NEG_TOL = 1e-6               # allow tiny negative due to FP
MAX_BAD_ELAST_SHARE = 0.05         # ≤5% of valid points may violate elasticity bounds
Y_FLOOR_RATIO = 0.01               # compute elasticity only where Y ≥ Y_floor
                                   # with Y_floor = max(EPS, Y_FLOOR_RATIO * median(Y>0))

OUTSIDE_FRAC = 0.15                # ±15% probes
# --------------------------------

def _parse_params(row):
    js = row.get("params_json")
    if pd.isna(js): return None
    try:
        arr = json.loads(js)
        return [float(x) for x in arr]
    except Exception:
        return None

def _parse_effective_controls(row):
    ec = row.get("effective_control_cols")
    if isinstance(ec, list): return ec
    if isinstance(ec, str):
        try:
            val = json.loads(ec)
            if isinstance(val, (list, tuple)): return list(val)
        except Exception:
            pass
    return []

def _ctrl_means(panel, cid, eff_cols):
    if not eff_cols: return {}
    df = panel.loc[panel[CLUSTER_COL]==cid, eff_cols] if isinstance(panel, pd.DataFrame) else pd.DataFrame()
    out = {}
    for c in eff_cols:
        out[c] = float(np.nanmean(df[c])) if c in df.columns else 0.0
    return out

def _x_band(panel, cid):
    if not isinstance(panel, pd.DataFrame) or X_COL not in panel.columns: return None, None, None, None
    s = panel.loc[panel[CLUSTER_COL]==cid, X_COL].dropna().astype(float)
    if s.empty: return None, None, None, None
    xmin, xmax = float(s.min()), float(s.max())
    xlo = float(np.percentile(s, X_CENTRAL_PCTS[0]))
    xhi = float(np.percentile(s, X_CENTRAL_PCTS[1]))
    return xmin, xmax, min(xlo, xhi), max(xlo, xhi)

def _grid(x0, x1):
    if not (np.isfinite(x0) and np.isfinite(x1)) or x1 <= x0: return None
    step_points = int(np.floor((x1 - x0) / DECIMAL_STEP)) + 1
    n = int(np.clip(step_points, 50, GRID_MAX))
    return np.linspace(x0, x1, n)

# ---------- predict & derivative (forms match your fitter) ----------
def _predict_Y(form, params, X, cm, eff_cols):
    X = np.asarray(X, dtype=float)

    if form in ("linear_X_controls","poly1"):
        const = params[0] if len(params)>0 else 0.0
        b1    = params[1] if len(params)>1 else 0.0
        y = const + b1*X
        for j,c in enumerate(eff_cols):
            idx = 2+j
            if idx < len(params): y = y + params[idx]*float(cm.get(c,0.0))
        return y

    if form.startswith("poly"):
        try: deg = int(form.replace("poly",""))
        except: deg = 1
        const = params[0] if len(params)>0 else 0.0
        y = np.full_like(X, const, dtype=float)
        for p in range(1, deg+1):
            idx = p
            if idx < len(params): y += params[idx]*(X**p)
        for j,c in enumerate(eff_cols):
            idx = 1+deg+j
            if idx < len(params): y += params[idx]*float(cm.get(c,0.0))
        return y

    if form in ("loglin","expdecay"):
        a = params[0] if len(params)>0 else 0.0
        b = params[1] if len(params)>1 else 0.0
        lin = a + b*X
        for j,c in enumerate(eff_cols):
            idx = 2+j
            if idx < len(params): lin += params[idx]*float(cm.get(c,0.0))
        return np.exp(lin)

    if form == "semilog":
        a = params[0] if len(params)>0 else 0.0
        b = params[1] if len(params)>1 else 0.0
        y = a + b*np.log(np.clip(X, EPS, None))
        for j,c in enumerate(eff_cols):
            idx = 2+j
            if idx < len(params): y += params[idx]*float(cm.get(c,0.0))
        return y

    if form == "doublelog":
        a = params[0] if len(params)>0 else 0.0
        b = params[1] if len(params)>1 else 0.0
        lnA = a
        for j,c in enumerate(eff_cols):
            idx = 2+j
            if idx < len(params): lnA += params[idx]*float(cm.get(c,0.0))
        return np.exp(lnA) * np.clip(X, EPS, None)**b

    if form == "reciprocal":
        a = params[0] if len(params)>0 else 0.0
        b = params[1] if len(params)>1 else 0.0
        y = a + b/np.clip(X, EPS, None)
        for j,c in enumerate(eff_cols):
            idx = 2+j
            if idx < len(params): y += params[idx]*float(cm.get(c,0.0))
        return y

    if form == "logistic":
        L  = params[0] if len(params)>0 else 1.0
        k  = params[1] if len(params)>1 else 1.0
        x0 = params[2] if len(params)>2 else 0.0
        shift = 0.0
        for j,c in enumerate(eff_cols):
            idx = 3+j
            if idx < len(params): shift += params[idx]*float(cm.get(c,0.0))
        eta = k*(X - (x0 + shift))
        return L / (1.0 + np.exp(-eta))

    return np.full_like(X, np.nan, dtype=float)

def _dYdX(form, params, X, Y, cm, eff_cols):
    X = np.asarray(X, dtype=float)

    if form in ("linear_X_controls","poly1"):
        return np.full_like(X, params[1] if len(params)>1 else 0.0, dtype=float)

    if form.startswith("poly"):
        try: deg = int(form.replace("poly",""))
        except: deg = 1
        d = np.zeros_like(X, dtype=float)
        for p in range(1, deg+1):
            idx = p
            if idx < len(params): d += p*params[idx]*(X**(p-1))
        return d

    if form in ("loglin","expdecay"):
        b = params[1] if len(params)>1 else 0.0
        return b*Y

    if form == "semilog":
        b = params[1] if len(params)>1 else 0.0
        return b/np.clip(X, EPS, None)

    if form == "doublelog":
        b = params[1] if len(params)>1 else 0.0
        return (b/np.clip(X, EPS, None))*Y

    if form == "reciprocal":
        b = params[1] if len(params)>1 else 0.0
        return -b/(np.clip(X, EPS, None)**2)

    if form == "logistic":
        L = params[0] if len(params)>0 else 1.0
        k = params[1] if len(params)>1 else 1.0
        return k * Y * (1.0 - Y/np.clip(L, EPS, None))

    return np.full_like(X, np.nan, dtype=float)

# ---------- guardrails ----------
def _logic_pass(form, params, gridX, cm, eff_cols):
    Y = _predict_Y(form, params, gridX, cm, eff_cols)
    d = _dYdX(form, params, gridX, Y, cm, eff_cols)
    if not (np.all(np.isfinite(Y)) and np.all(np.isfinite(d))): return False, np.nan
    neg_share = float(np.mean(d < -DERIV_TOL))
    return (neg_share <= MAX_NEG_DERIV_SHARE), neg_share

def _stability_pass(form, params, gridX, cm, eff_cols):
    Y = _predict_Y(form, params, gridX, cm, eff_cols)
    d = _dYdX(form, params, gridX, Y, cm, eff_cols)
    if not (np.all(np.isfinite(Y)) and np.all(np.isfinite(d))): return False, np.nan, np.nan, np.nan

    # use elasticity only where Y is “meaningful”
    posY = Y[Y > 0]
    if posY.size == 0:
        return False, np.nan, np.nan, np.nan
    y_floor = max(EPS, np.median(posY) * Y_FLOOR_RATIO)
    mask = Y >= y_floor
    if not np.any(mask):
        return False, np.nan, np.nan, np.nan

    eps = (d[mask] * (gridX[mask] / np.clip(Y[mask], EPS, None)))
    bad_share = float(np.mean((eps < -ELAST_NEG_TOL) | (eps > ELAST_HI + 1e-12)))
    p5, p95 = float(np.percentile(eps, 5)), float(np.percentile(eps, 95))
    return (bad_share <= MAX_BAD_ELAST_SHARE), bad_share, p5, p95

def _sensible_pass(form, params, xmin, xmax, cm, eff_cols):
    if not (np.isfinite(xmin) and np.isfinite(xmax)) or xmax <= 0:
        return True, np.nan, np.nan
    Xprobe = np.array([max(EPS, xmin*(1.0-OUTSIDE_FRAC)), xmax*(1.0+OUTSIDE_FRAC)], dtype=float)
    Y = _predict_Y(form, params, Xprobe, cm, eff_cols)
    d = _dYdX(form, params, Xprobe, Y, cm, eff_cols)
    if not (np.all(np.isfinite(Y)) and np.all(np.isfinite(d))):
        return False, np.nan, np.nan
    eps = d * (Xprobe / np.clip(Y, EPS, None))
    ok = np.all(Y >= -1e-12) and np.all(eps >= -ELAST_NEG_TOL) and np.all(eps <= ELAST_HI + 1e-12)
    return bool(ok), float(np.min(eps)), float(np.max(eps))

def _apply_metric_gates(df):
    df["mae_q"]  = pd.to_numeric(df.get("MAE"), errors="coerce")
    df["rmse_q"] = pd.to_numeric(df.get("RMSE"), errors="coerce")
    df["r2_q"]   = pd.to_numeric(df.get("R2"), errors="coerce")
    df["gate_pass"] = False
    for (cid, tgt), grp in df.groupby([CLUSTER_COL, "target"], dropna=False):
        best_rmse = np.nanmin(grp["rmse_q"].to_numpy())
        best_mae  = np.nanmin(grp["mae_q"].to_numpy())
        for idx in grp.index:
            r2 = df.at[idx, "r2_q"]; rm = df.at[idx, "rmse_q"]; ma = df.at[idx, "mae_q"]
            gate = (np.isfinite(rm) and rm <= 1.3*best_rmse) and \
                   (np.isfinite(ma) and ma <= 1.3*best_mae) and \
                   (np.isfinite(r2) and r2 >= 0.0)
            df.at[idx, "gate_pass"] = bool(gate)

def apply_incrementality_guardrails_robust(results_df, panel):
    out = results_df.copy()
    out["logic_pass"]     = False
    out["stability_pass"] = False
    out["sensible_pass"]  = False
    out["gate_pass"]      = False
    out["survivor"]       = False
    out["fail_reason"]    = ""

    # diagnostics
    out["neg_deriv_share"] = np.nan
    out["elast_bad_share"] = np.nan
    out["elast_p5"] = np.nan
    out["elast_p95"] = np.nan
    out["outside_eps_min"] = np.nan
    out["outside_eps_max"] = np.nan

    # pre-slice panel
    p_by_c = dict(tuple(panel.groupby(CLUSTER_COL, dropna=False))) if isinstance(panel, pd.DataFrame) else {}

    for i, row in out.iterrows():
        form = row["form"]
        params = _parse_params(row)
        eff_cols = _parse_effective_controls(row)
        cid = row[CLUSTER_COL]
        if not params:
            out.at[i,"fail_reason"] = "no_params"; continue
        dfc = p_by_c.get(cid)
        if dfc is None or dfc.empty or X_COL not in dfc.columns:
            out.at[i,"fail_reason"] = "no_train_data"; continue

        xmin, xmax, xlo, xhi = _x_band(panel, cid)
        if any(v is None for v in [xmin,xmax,xlo,xhi]) or not (xhi > xlo):
            out.at[i,"fail_reason"] = "bad_range"; continue
        gridX = _grid(xlo, xhi)
        if gridX is None:
            out.at[i,"fail_reason"] = "bad_range"; continue

        cm = _ctrl_means(panel, cid, eff_cols)

        lp, neg_share = _logic_pass(form, params, gridX, cm, eff_cols)
        out.at[i,"logic_pass"] = lp
        out.at[i,"neg_deriv_share"] = neg_share
        if not lp:
            out.at[i,"fail_reason"] = "logic"; continue

        sp, bad_share, p5, p95 = _stability_pass(form, params, gridX, cm, eff_cols)
        out.at[i,"stability_pass"] = sp
        out.at[i,"elast_bad_share"] = bad_share
        out.at[i,"elast_p5"] = p5
        out.at[i,"elast_p95"] = p95
        if not sp:
            out.at[i,"fail_reason"] = "stability"; continue

        bp, eps_min, eps_max = _sensible_pass(form, params, xmin, xmax, cm, eff_cols)
        out.at[i,"sensible_pass"] = bp
        out.at[i,"outside_eps_min"] = eps_min
        out.at[i,"outside_eps_max"] = eps_max
        if not bp:
            out.at[i,"fail_reason"] = "sensible"; continue

    # metric gates
    out["survivor"] = out["logic_pass"] & out["stability_pass"] & out["sensible_pass"]

    # Final gate (guardrails only)
    out["survivor"] = out["logic_pass"] & out["stability_pass"] & out["sensible_pass"]

    # Return ALL curves that pass the guardrails (no fit metrics used here)
    best_models_df = out.loc[out["survivor"]].copy()

    # tidy ordering (optional)
    out = out.sort_values([CLUSTER_COL, "target", "form"]).reset_index(drop=True)
    best_models_df = best_models_df.sort_values([CLUSTER_COL, "target", "form"]).reset_index(drop=True)

    return out, best_models_df

# ---- run ----
results_guardrailed_df, best_models_df = apply_incrementality_guardrails_robust(results_df, panel)

# quick summary like before
print("Number of models passing each criterion:")
print("-"*60)
print("Logic Check Passed:", int(results_guardrailed_df["logic_pass"].sum()))
print("Stability Check Passed:", int(results_guardrailed_df["stability_pass"].sum()))
print("Sensible Behavior Check Passed:", int(results_guardrailed_df["sensible_pass"].sum()))
print("Final Gate (All Criteria) Passed:", int(results_guardrailed_df["survivor"].sum()))
print("-"*60)

display(results_guardrailed_df.head(20))


Number of models passing each criterion:
------------------------------------------------------------
Logic Check Passed: 509
Stability Check Passed: 465
Sensible Behavior Check Passed: 308
Final Gate (All Criteria) Passed: 308
------------------------------------------------------------


,form,equation,params_json,MAE,RMSE,R2,n,effective_control_cols,cluster_id,target,error,logic_pass,stability_pass,sensible_pass,gate_pass,survivor,fail_reason,neg_deriv_share,elast_bad_share,elast_p5,elast_p95,outside_eps_min,outside_eps_max
0,doublelog,Q = exp(4.63703 -1.56585·seasonality -0.299261...,"[4.637030607124182, 0.004616605199437726, -1.5...",103.689189,111.759394,-6.183562,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,False,False,False,False,stability,0.0,NaN,NaN,NaN,NaN,NaN
1,expdecay,Q = α + β·(1 - exp(-k·P)) + Σ(γ_i·Control_i),None,NaN,NaN,NaN,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,Parameter count mismatch for control terms in ...,False,False,False,False,False,no_params,NaN,NaN,NaN,NaN,NaN,NaN
2,linear_X_controls,Q = 103.237 +0.00461661·P -1.56585·seasonality...,"[103.2373397742181, 0.004616605199437726, -1.5...",17.026101,23.262695,0.688763,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.341801,0.745651,1.358588e-01,8.068153e-01
3,logistic,Q = 225.556 / (1 + exp(-9.0974e-05·(P - (-1301...,"[225.5555717258971, 9.097401672758972e-05, -13...",16.758923,23.062769,0.694089,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.319704,0.692634,1.051800e-01,3.269661e-01
4,loglin,Q = exp(4.38995 +3.72319e-05·P -0.0505899·seas...,"[4.389950356204962, 3.723193158795367e-05, -0....",18.138194,24.371224,0.658393,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.185440,1.046870,5.614235e-02,1.491381e+00
5,poly2,Q = 102.11 +0.0054406·P -2.49976e-08·P^2 -1.27...,"[102.11031857478781, 0.005440602461672165, -2....",17.028808,23.206764,0.690257,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.394005,0.658155,1.710526e-01,6.347041e-01
6,poly3,Q = 115.138 +0.00310095·P +1.43735e-07·P^2 -3....,"[115.13780795651412, 0.003100948525312037, 1.4...",16.834269,23.134931,0.692172,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,False,False,False,sensible,0.0,0.0,0.321727,0.690791,-3.313581e-01,1.004773e-01
7,poly4,Q = 6.51448e-06 +0.0204643·P -1.81023e-06·P^2 ...,"[6.51448258199669e-06, 0.020464263503491755, -...",17.395821,23.382810,0.685540,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,False,False,False,sensible,0.0,0.0,0.385608,1.018141,-4.303968e+01,8.641265e-01
8,poly5,Q = 1.24365e-13 -7.12329e-08·P +3.06465e-06·P^...,"[1.2436466729364595e-13, -7.123290808238266e-0...",20.735628,26.545772,0.594713,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,False,False,False,sensible,0.0,0.0,0.210466,1.489035,-6.525888e+12,1.839946e+00
9,reciprocal,Q = 224.636 -276634 / P +8.20753·seasonality -...,"[224.6356892715859, -276634.224070696, 8.20753...",24.344635,31.647917,0.423948,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,False,False,False,sensible,0.0,0.0,0.078753,0.701054,5.401237e-02,1.834556e+11


In [15]:
# In-Sample Performance Test

df = results_guardrailed_df.loc[results_guardrailed_df["survivor"]].copy()

# ensure numeric
for c in ["RMSE", "MAE", "R2"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# per-cluster×target best RMSE/MAE
mins = (
    df.groupby([CLUSTER_COL, "target"], dropna=False)[["RMSE", "MAE"]]
      .min()
      .rename(columns={"RMSE": "best_in_cluster_RMSE", "MAE": "best_in_cluster_MAE"})
      .reset_index()
)

# attach and apply the 30% fit gate
df = df.merge(mins, on=[CLUSTER_COL, "target"], how="left")
df["in_sample_passed"] = (
    df["RMSE"].le(1.3 * df["best_in_cluster_RMSE"]) &
    df["MAE"].le(1.3 * df["best_in_cluster_MAE"]) &
    df["R2"].ge(0)
).fillna(False)

display(df.head(50))

,form,equation,params_json,MAE,RMSE,R2,n,effective_control_cols,cluster_id,target,error,logic_pass,stability_pass,sensible_pass,gate_pass,survivor,fail_reason,neg_deriv_share,elast_bad_share,elast_p5,elast_p95,outside_eps_min,outside_eps_max,best_in_cluster_RMSE,best_in_cluster_MAE,in_sample_passed
0,linear_X_controls,Q = 103.237 +0.00461661·P -1.56585·seasonality...,"[103.2373397742181, 0.004616605199437726, -1.5...",17.026101,23.262695,6.887625e-01,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.341801,0.745651,0.135859,0.806815,23.062769,16.758923,True
1,logistic,Q = 225.556 / (1 + exp(-9.0974e-05·(P - (-1301...,"[225.5555717258971, 9.097401672758972e-05, -13...",16.758923,23.062769,6.940893e-01,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.319704,0.692634,0.105180,0.326966,23.062769,16.758923,True
2,loglin,Q = exp(4.38995 +3.72319e-05·P -0.0505899·seas...,"[4.389950356204962, 3.723193158795367e-05, -0....",18.138194,24.371224,6.583932e-01,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.185440,1.046870,0.056142,1.491381,23.062769,16.758923,True
3,poly2,Q = 102.11 +0.0054406·P -2.49976e-08·P^2 -1.27...,"[102.11031857478781, 0.005440602461672165, -2....",17.028808,23.206764,6.902574e-01,148,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_au_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.394005,0.658155,0.171053,0.634704,23.062769,16.758923,True
4,linear_X_controls,Q = 126.69 +0.0208206·P +5.63561·seasonality +...,"[126.69045190815626, 0.02082061464190946, 5.63...",10.721856,13.341810,7.010088e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.286245,0.635399,0.115338,0.785346,13.338001,10.697938,True
5,logistic,Q = 167.868 / (1 + exp(-0.000558097·(P - (-192...,"[167.86842670746276, 0.0005580973216762727, -1...",10.755595,13.402386,6.982877e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.259842,0.687950,0.089807,0.407261,13.338001,10.697938,True
6,loglin,Q = exp(5.34401 +0.000214507·P +0.05226·season...,"[5.344008236925215, 0.00021450658440332095, 0....",11.502183,14.364019,6.534381e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.137960,0.599505,0.044850,1.258597,13.338001,10.697938,True
7,poly2,Q = 125.449 +0.0221105·P -2.97517e-07·P^2 +5.6...,"[125.44904067378488, 0.022110524198526958, -2....",10.698263,13.338118,7.011743e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.301033,0.622552,0.124464,0.719614,13.338001,10.697938,True
8,poly3,Q = 125.282 +0.0226486·P -5.78749e-07·P^2 +3.8...,"[125.28161027708056, 0.022648580283224605, -5....",10.697938,13.338001,7.011795e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.304352,0.619584,0.127465,0.762684,13.338001,10.697938,True
9,semilog,Q = -24.6807 +27.8856·ln(P) +3.8438·seasonalit...,"[-24.680650054161493, 27.88556508518539, 3.843...",11.452591,14.166905,6.628844e-01,124,"[seasonality, Unit Voucher Spend Acquired Cust...",affiliate_dn_de_acquisition,A_acq,NaN,True,True,True,False,True,,0.0,0.0,0.326933,0.629092,0.263132,2.146183,13.338001,10.697938,True


In [16]:
# Hold-out (TEST) evaluation + winner selection

# Safety checks
if "panel_test" not in globals() or panel_test is None or panel_test.empty:
    raise RuntimeError("panel_test is missing or empty. Make sure you built it with make_panel(...) from the TEST split.")

if "df" not in globals() or df is None or df.empty:
    raise RuntimeError("df is missing or empty. This cell expects the in-sample summary DataFrame created right after guardrails.")

# We only evaluate models that both survived guardrails AND passed the in-sample fit gate
candidates = df.loc[df["in_sample_passed"]].copy()
if candidates.empty:
    raise RuntimeError("No candidates with in_sample_passed == True. Loosen the in-sample gate or inspect prior steps.")

# Map your tag -> actual Y column in panel_test
TAG_TO_YCOL = {
    "A_acq": "_Y_acq",
    "B_rea": "_Y_rea",
    # "C_total": "_Y_total",  # uncomment if you compute total as well
}

def _tag_to_ycol(tag: str) -> str:
    ycol = TAG_TO_YCOL.get(str(tag))
    if ycol is None:
        raise ValueError(f"Unrecognized target tag '{tag}'. Update TAG_TO_YCOL mapping.")
    return ycol

# Evaluate every candidate on hold-out TEST data
holdout_rows = []

group_cols = [CLUSTER_COL, "target"]
for (cid, target), g in candidates.groupby(group_cols, dropna=False):
    ycol = _tag_to_ycol(target)

    # TEST slice for this cluster
    gtest = panel_test.loc[panel_test[CLUSTER_COL] == cid]
    if gtest.empty or ycol not in gtest.columns or "X" not in gtest.columns:
        # Nothing to evaluate; keep a record with NaNs
        for _, row in g.iterrows():
            holdout_rows.append({
                CLUSTER_COL: cid,
                "target": target,
                "form": row["form"],
                "holdout_n": 0,
                "holdout_RMSE": np.nan,
                "holdout_MAE": np.nan,
            })
        continue

    # Controls actually used by this model (effective at fit-time)
    # and their TEST means for this cluster
    for _, row in g.iterrows():
        try:
            form     = row["form"]
            params   = _parse_params(row)                       # -> list[float]
            eff_cols = _parse_effective_controls(row)           # -> list[str]
            cm       = _ctrl_means(panel_test, cid, eff_cols)   # TEST means for this cluster

            X = gtest["X"].astype(float).values
            y = gtest[ycol].astype(float).values

            # Predict with the exact same form & params used in TRAIN, but evaluated on TEST X and TEST control means
            yhat = _predict_Y(form, params, X, cm, eff_cols)

            mae, rmse, _ = score_fit(y, yhat)
            holdout_rows.append({
                CLUSTER_COL: cid,
                "target": target,
                "form": form,
                "holdout_n": int(len(y)),
                "holdout_RMSE": float(rmse),
                "holdout_MAE": float(mae),
            })
        except Exception as e:
            # Keep a row so you can debug later
            holdout_rows.append({
                CLUSTER_COL: cid,
                "target": target,
                "form": row.get("form"),
                "holdout_n": int(gtest.shape[0]),
                "holdout_RMSE": np.nan,
                "holdout_MAE": np.nan,
                "error": str(e),
            })

holdout_scores = pd.DataFrame(holdout_rows)

# Keep only evaluated candidates (inner-join on cluster/target/form), then select winners
merged = (
    candidates.merge(holdout_scores, on=[CLUSTER_COL, "target", "form"], how="inner")
)

# Rank within each (cluster, target): primary key = lowest holdout_RMSE, tie-break by holdout_MAE
winners = (
    merged
    .sort_values([CLUSTER_COL, "target", "holdout_RMSE", "holdout_MAE"], na_position="last")
    .groupby([CLUSTER_COL, "target"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# Nice, compact columns to keep (add/remove as you like)
cols_to_show = [
    CLUSTER_COL, "target", "form",
    "equation", "params_json", "effective_control_cols",
    "n", "RMSE", "MAE", "R2",            # in-sample metrics (for reference)
    "holdout_n", "holdout_RMSE", "holdout_MAE",
    "in_sample_passed", "survivor",
]
cols_to_show = [c for c in cols_to_show if c in winners.columns]

print(f"Evaluated on TEST and selected winners for {winners[[CLUSTER_COL,'target']].drop_duplicates().shape[0]} (cluster, target) pairs.")
display(holdout_scores.sort_values([CLUSTER_COL, "target", "holdout_RMSE", "holdout_MAE"]))
print("-" * 80)
print("Final winners (one per cluster & target):")
display(winners[cols_to_show])

winners.to_csv("holdout_winners.csv", index=False)
print("Saved: holdout_winners.csv")


Evaluated on TEST and selected winners for 63 (cluster, target) pairs.


,cluster_id,target,form,holdout_n,holdout_RMSE,holdout_MAE
1,affiliate_dn_au_acquisition,A_acq,logistic,37,22.716300,20.882330
3,affiliate_dn_au_acquisition,A_acq,poly2,37,24.071228,22.357186
0,affiliate_dn_au_acquisition,A_acq,linear_X_controls,37,24.183991,22.525208
2,affiliate_dn_au_acquisition,A_acq,loglin,37,28.929785,27.267280
7,affiliate_dn_de_acquisition,A_acq,poly2,31,6.161744,4.670762
...,...,...,...,...,...,...
260,sms_ms_nl_reactivation,B_rea,logistic,30,38.862828,35.246098
259,sms_ms_nl_reactivation,B_rea,linear_X_controls,30,89.389810,70.644042
261,sms_ms_nl_reactivation,B_rea,poly2,30,103.539739,74.270203
263,sms_ms_us_reactivation,B_rea,logistic,22,1.388741,1.387532


--------------------------------------------------------------------------------
Final winners (one per cluster & target):


,cluster_id,target,form,equation,params_json,effective_control_cols,n,RMSE,MAE,R2,holdout_n,holdout_RMSE,holdout_MAE,in_sample_passed,survivor
0,affiliate_dn_au_acquisition,A_acq,logistic,Q = 225.556 / (1 + exp(-9.0974e-05·(P - (-1301...,"[225.5555717258971, 9.097401672758972e-05, -13...","[seasonality, Unit Voucher Spend Acquired Cust...",148,23.062769,16.758923,0.694089,37,22.716300,20.882330,True,True
1,affiliate_dn_de_acquisition,A_acq,poly2,Q = 125.449 +0.0221105·P -2.97517e-07·P^2 +5.6...,"[125.44904067378488, 0.022110524198526958, -2....","[seasonality, Unit Voucher Spend Acquired Cust...",124,13.338118,10.698263,0.701174,31,6.161744,4.670762,True,True
2,affiliate_dn_nl_acquisition,A_acq,linear_X_controls,Q = 14.0208 +0.01262·P +2.18457·seasonality -0...,"[14.020839768999902, 0.012620007569101099, 2.1...","[seasonality, Unit Voucher Spend Acquired Cust...",123,5.625468,4.071541,0.555809,31,4.107544,3.323752,True,True
3,affiliate_dn_us_acquisition,A_acq,linear_X_controls,Q = 521.808 +0.0198428·P +5.64971·seasonality ...,"[521.8077987330893, 0.019842782740505053, 5.64...","[seasonality, Unit Voucher Spend Acquired Cust...",149,154.773702,94.784015,0.734637,37,49.779570,42.255644,True,True
4,affiliate_ms_au_acquisition,A_acq,loglin,Q = exp(5.22522 +2.67708e-05·P +0.0440605·seas...,"[5.225219068503952, 2.677082283416306e-05, 0.0...","[seasonality, Unit Voucher Spend Acquired Cust...",149,20.668856,16.224203,0.801653,37,16.619671,13.829834,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,sms_ms_au_reactivation,B_rea,linear_X_controls,Q = -593.805 +0.0569727·P +49.6519·seasonality...,"[-593.8046095393967, 0.056972668285511854, 49....","[seasonality, Unit Voucher Spend Reactivated C...",119,348.761445,236.681442,0.503904,30,199.423345,185.648139,True,True
59,sms_ms_be_reactivation,B_rea,doublelog,Q = exp(0.914104 +1.01643·seasonality -0.06395...,"[0.914104193478562, 0.5095379912887988, 1.0164...","[seasonality, Unit Voucher Spend Reactivated C...",113,19.294535,10.162362,0.760347,29,19.871587,10.992907,True,True
60,sms_ms_de_reactivation,B_rea,logistic,Q = 1434.66 / (1 + exp(-0.000346261·(P - (2289...,"[1434.659249650746, 0.0003462606085380631, 228...","[seasonality, Unit Voucher Spend Reactivated C...",118,145.983139,91.380830,0.761753,29,51.411937,47.479596,True,True
61,sms_ms_nl_reactivation,B_rea,logistic,Q = 910.657 / (1 + exp(-0.000603198·(P - (1329...,"[910.6568569702863, 0.0006031981493454324, 132...","[seasonality, Unit Voucher Spend Reactivated C...",117,125.788450,69.203977,0.751614,30,38.862828,35.246098,True,True


Saved: holdout_winners.csv


In [17]:
from google.colab import files
files.download("holdout_winners.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# ================================================
# Plot all winning curves on hold-out data
# ================================================

import matplotlib.pyplot as plt

# Ensure these are loaded
assert 'winners' in globals(), "You need the winners DataFrame from the hold-out step."
assert 'panel_test' in globals(), "You need the hold-out dataset (panel_test)."

# Sort for consistent plotting
winners_sorted = winners.sort_values([CLUSTER_COL, "target"]).reset_index(drop=True)

for _, row in winners_sorted.iterrows():
    cid = row[CLUSTER_COL]
    target = row["target"]
    form = row["form"]

    # parse params and effective controls
    params = _parse_params(row)
    eff_cols = _parse_effective_controls(row)
    cm = _ctrl_means(panel_test, cid, eff_cols)

    # select hold-out data for this cluster
    gtest = panel_test.loc[panel_test[CLUSTER_COL] == cid]
    ycol = TAG_TO_YCOL[target]
    if gtest.empty or ycol not in gtest.columns:
        continue

    # Actual data (scatter)
    X_data = gtest["X"].astype(float).values
    y_data = gtest[ycol].astype(float).values

    # Predicted curve (smooth line)
    X_pred = np.linspace(X_data.min(), X_data.max(), 100)
    y_pred = _predict_Y(form, params, X_pred, cm, eff_cols)

    plt.figure(figsize=(6,4))
    plt.scatter(X_data, y_data, color='gray', alpha=0.7, label="Hold-out data")
    plt.plot(X_pred, y_pred, color='blue', linewidth=2, label=f"Predicted ({form})")
    plt.title(f"{cid} — {target}\nEquation: {form}")
    plt.xlabel("Media Spend (X)")
    plt.ylabel("Target (Y)")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


Output hidden; open in https://colab.research.google.com to view.